In [1]:
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

from sklearn.neural_network import MLPRegressor

from model import single_ml_model_exp, grid_search_exp

import config
%load_ext autoreload
%autoreload 2
    
model = MLPRegressor(activation='logistic', solver='lbfgs') 

experiment_id = 'chamados'
model_name = 'mlp'
normalize = True
# Tarefa 5.1: force=True para regenerar -- o .pkl persistido foi treinado com
# diff_kpss=True (config antiga), mas o codigo acima usa diff_kpss=False
# desde o commit 43dae50 (2026-07-02, mesmo commit que corrigiu activation em
# arima_mlp.ipynb na Tarefa 3.9, mudanca ja aprovada e documentada no
# CLAUDE.md Secao 3.5). Sem force=True, grid_seach_multiple_bases pularia as
# series ja existentes (idempotencia intencional, CLAUDE.md Secao 3.2) e o
# .pkl continuaria desatualizado.
force = True
model_exec = 10

experiment_params = {
    'diff_kpss': False,
    'horizon': 1,
    'type_filter': None
}

model_parameters = {
    'hidden_layer_sizes': [10, 20, 50], 
    'max_iter': [1000],
}

# Tarefa 5.1: lista explicita das 17 series da baseline classica, em vez de
# grid_seach_multiple_bases(...) (que depende de config.BASE_NAME_LIST -- hoje
# com so 4 series ativas, as mesmas de FS_DEV_SERIES). Mesmo padrao ja usado
# em arima_mlp.ipynb (Tarefa 3.9), para nao alterar um arquivo de
# configuracao compartilhado por outros notebooks so para esta regeneracao
# pontual do baseline.
baseline_series_list = [
    'airlines.txt', 'ausbee.txt', 'austres.txt', 'coloradoRiver.txt',
    'gasoline.txt', 'heartrate.txt', 'lakeerie.txt', 'lynx.txt', 'milk.txt',
    'ozon.txt', 'pollution.txt', 'redwine.txt', 'sunspot.txt', 'taylor.txt',
    'temperature.txt', 'Unemployment.txt', 'woolyrnq.txt',
]

for base_name in baseline_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()


Failed to read module file 'C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen impor

airlines.txt
{'hidden_layer_sizes': 20, 'max_iter': 1000}
ausbee.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
austres.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
coloradoRiver.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
gasoline.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
heartrate.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
lakeerie.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
lynx.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
milk.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
ozon.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
pollution.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
redwine.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
sunspot.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
taylor.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
temperature.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
Unemployment.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
woolyrnq.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}


In [ ]:
# === CELULA DE VERIFICACAO POS-EXECUCAO -- Tarefa 5.1 ===
# Roda automaticamente ao final deste "Run All" (ou pode ser reexecutada
# isoladamente depois). Confirma: (a) o .pkl regenerado persiste
# diff_kpss=False; (b) comparado ao snapshot de hash pre-regeneracao, SO os
# 17 *_1mlp.pkl mudaram -- nenhum outro modelo (_1arima/_1amv1/_1svr/_1as)
# foi tocado.
import hashlib
import pickle
from pathlib import Path

chamados_dir = Path(config.ROOT_PATH) / 'data' / 'result' / 'chamados'

print("--- (a) diff_kpss persistido apos a regeneracao ---")
for base_name in ['airlines.txt', 'sunspot.txt']:
    series = base_name.replace('.txt', '')
    with open(chamados_dir / f'{series}_1mlp.pkl', 'rb') as f:
        saved = pickle.load(f)
    exp_params = saved[0]['experiment'].experiment_params
    print(f"{series}: diff_kpss={exp_params['diff_kpss']!r}")
    assert exp_params['diff_kpss'] is False, f"{series}: esperado False, achou {exp_params['diff_kpss']!r}"
print("OK -- diff_kpss=False confirmado nos .pkl regenerados.")

print()
print("--- (b) comparando hashes com o snapshot pre-regeneracao ---")
snapshot_path = Path(config.ROOT_PATH) / 'data' / 'result' / 'chamados_1mlp_pkl_hashes_pre_tarefa_5_1_20260721.txt'
old_hashes = {}
for line in snapshot_path.read_text(encoding='utf-8-sig').splitlines():
    if not line.strip():
        continue
    h, path = line.split('  ', 1)
    old_hashes[Path(path.strip()).name] = h.strip()

changed, unchanged = [], []
for pkl_path in sorted(chamados_dir.glob('*.pkl')):
    new_hash = hashlib.sha256(pkl_path.read_bytes()).hexdigest().upper()
    old_hash = old_hashes.get(pkl_path.name)
    if old_hash is None:
        print(f"AVISO: {pkl_path.name} nao estava no snapshot (arquivo novo?)")
        continue
    (changed if new_hash != old_hash else unchanged).append(pkl_path.name)

print(f"Arquivos que MUDARAM: {len(changed)}")
for name in sorted(changed):
    print(f"  {name}")
print(f"Arquivos que permaneceram IGUAIS: {len(unchanged)}")

only_1mlp_changed = all(name.endswith('_1mlp.pkl') for name in changed)
print()
if only_1mlp_changed and len(changed) == 17:
    print("OK -- exatamente os 17 *_1mlp.pkl mudaram, nada mais.")
else:
    print("ATENCAO -- o conjunto de arquivos alterados nao bate com o esperado "
          "(deveria ser exatamente os 17 *_1mlp.pkl). Revisar antes de prosseguir.")


--- (a) diff_kpss persistido apos a regeneracao ---
airlines: diff_kpss=False
sunspot: diff_kpss=False
OK -- diff_kpss=False confirmado nos .pkl regenerados.

--- (b) comparando hashes com o snapshot pre-regeneracao ---
Arquivos que MUDARAM: 18
  Unemployment_1mlp.pkl
  airlines_1amv1.pkl
  airlines_1mlp.pkl
  ausbee_1mlp.pkl
  austres_1mlp.pkl
  coloradoRiver_1mlp.pkl
  gasoline_1mlp.pkl
  heartrate_1mlp.pkl
  lakeerie_1mlp.pkl
  lynx_1mlp.pkl
  milk_1mlp.pkl
  ozon_1mlp.pkl
  pollution_1mlp.pkl
  redwine_1mlp.pkl
  sunspot_1mlp.pkl
  taylor_1mlp.pkl
  temperature_1mlp.pkl
  woolyrnq_1mlp.pkl
Arquivos que permaneceram IGUAIS: 67

ATENCAO -- o conjunto de arquivos alterados nao bate com o esperado (deveria ser exatamente os 17 *_1mlp.pkl). Revisar antes de prosseguir.
